# Project geolocation: bulk infrastructure by state

Read the project's `state` column, group rows by that value, fetch infrastructure for each state, then match only against that state's cache. The default input is Dominion, but the same steps work for a CSV containing multiple states.

No new libraries are used. `groupby("state")` splits the table into one group per state. `fetch_power_infrastructure(state)` uses that group's code in the Overpass area selector, with two bulk queries per state. Raw results live in `data/raw/osm/<STATE>/`; generated project outputs live in `data/processed/geo/`.

References: [Overpass query language](https://wiki.openstreetmap.org/wiki/Overpass_API/Overpass_QL), [OSM attribution](https://www.openstreetmap.org/copyright).

In [ ]:
import json
import pandas as pd

from gridlock.paths import PROCESSED_DATA_DIR
from gridlock.geolocation import (
    OSM_DIR, GEO_DIR, normalize_state, fetch_power_infrastructure,
    find_project_matches, build_project_geometry, save_results,
)

# To process another dataset (including multiple states), change only this path.
PROJECTS_PATH = PROCESSED_DATA_DIR / "dominion" / "dominion_projects.csv"
OUTPUT_PREFIX = PROJECTS_PATH.stem.removesuffix("_projects").removesuffix("_project_list")

## 1. Read and validate project states
The source ingestion notebooks assign `SC` to the Dominion dataset and `GA` to the Georgia dataset. This notebook never supplies a fallback state: a missing or malformed value stops processing. Whitespace and lowercase letters are normalized.

The state is the project's planning jurisdiction; it does not assert that every part of a cross-border line lies inside that state. Read IDs as strings and preserve repeated IDs using a state/source/page/ID key.

In [ ]:
df_projects = pd.read_csv(PROJECTS_PATH, dtype=str, keep_default_na=False)
for column in ["state", "project_id", "project_name"]:
    if column not in df_projects:
        raise ValueError("Missing required project column: " + column)
if df_projects.empty:
    raise ValueError("The project CSV is empty")
df_projects["state"] = df_projects["state"].apply(normalize_state)
projects = df_projects.to_dict("records")
for project in projects:
    source_file = project.get("source_file", PROJECTS_PATH.name)
    source_page = project.get("source_page", project.get("source_pdf_page", ""))
    project["project_key"] = project["state"] + ":" + source_file + ":" + source_page + ":" + project["project_id"]
assert len({p["project_key"] for p in projects}) == len(projects), "Duplicate source rows need review"
df_projects = pd.DataFrame(projects)
print("Total projects:", len(df_projects))
print(df_projects.groupby("state").size().to_string())

## 2. Fetch state infrastructure and preserve its geometry
For each state in the CSV, query `power=substation` and `power=line` using the `US-<state>` administrative area. Include all operators so a new utility or state needs no query-code changes. Utility/operator agreement is checked during matching instead of using a fixed Dominion-only query.

`nwr` includes nodes, ways, and relations; `out geom` returns coordinates. Raw responses and query files are cached in separate state folders. Existing successful responses are reused. The older Dominion-only cache is preserved but is not reused as if it contained all statewide infrastructure.

GeoJSON coordinates are [longitude, latitude]. Preserve full line vertices and substation polygons. Relation members remain GeometryCollections. The feature's `state` records which state's bulk query returned it, including ways crossing that state's boundary. No centroid distances are calculated.

In [ ]:
infrastructure_by_state = {}
for state, state_projects in df_projects.groupby("state"):
    infrastructure_by_state[state] = fetch_power_infrastructure(state)
    features = infrastructure_by_state[state]
    print(state, "projects:", len(state_projects), "OSM features:", len(features))
    print("  Raw cache:", OSM_DIR / state)

## 3. Inspect state coverage
An absent name, missing operator tag, future construction, or endpoint outside the queried state can leave projects unresolved. A state is never inferred from a project name or silently borrowed from another group.

In [ ]:
for state, features in infrastructure_by_state.items():
    print(state)
    print("  Substations:", sum(f["properties"]["power"] == "substation" for f in features))
    print("  Lines:", sum(f["properties"]["power"] == "line" for f in features))
    print("  Unnamed:", sum(not f["properties"]["osm_feature_name"] for f in features))
    assert all(f["properties"]["state"] == state for f in features)

## 4. Find candidates locally
`normalize_name()` removes case/punctuation differences, voltage labels, and words such as ?substation.? `find_project_matches()` compares title endpoints and names mentioned in descriptions against OSM names and aliases.

- **High**: one matching endpoint feature, matching voltage, corroborated utility/operator, and the same endpoint mentioned in the description. Still requires confirmation of project scope.
- **Medium**: a direct normalized title/endpoint name match without enough corroboration.
- **Low**: a description-only mention, overlapping name words, competing matches, a voltage conflict, or missing geometry.

`fuzzy_name` here means a simple overlap of at least two words, not a new library or an opaque score. Similar spelling alone never confirms a match. A PDF replacement character may obscure a separator; these candidates are not promoted to high confidence.

An OSM feature may be an existing line referenced by a construction project rather than the construction footprint itself. Every candidate keeps its geometry, evidence, and OSM link for review.

Match within each state group. The matching function also rejects features whose state differs, even if they have an identical name.

In [ ]:
all_matches = []
matches_by_project = {}
for state, state_projects in df_projects.groupby("state"):
    features = infrastructure_by_state[state]
    for project in state_projects.to_dict("records"):
        matches = find_project_matches(project, features)
        matches_by_project[project["project_key"]] = matches
        all_matches.extend(matches)

print("Candidate matches:", len(all_matches))
if all_matches:
    df_candidates = pd.DataFrame(all_matches)
    print(df_candidates.groupby(["state", "match_confidence"]).size().to_string())
    print(df_candidates[["state", "project_key", "project_name", "osm_feature_name", "match_confidence"]].head(20).to_string(index=False))

## 5. Confirm reviewed matches
Keep `CONFIRMED_MATCHES` empty until the candidate's actual location and project role have been checked. Review the saved candidate CSV and its OSM links. Confirming means the feature represents the intended project site/endpoint, not merely a similar name.

Add entries in this form, substituting real IDs and notes:

```python
CONFIRMED_MATCHES = {
    "STATE:source_file:source_page:project_id": [
        {"osm_feature_id": "way/OSM_ID", "note": "Evidence that this is the intended endpoint..."},
    ],
}
```

For a two-endpoint project, confirm both endpoints. For an OSM line, the note must explain why its extent represents this project's scope. Candidate confidence and confirmation are separate: even a high-confidence candidate is not silently accepted.

In [ ]:
CONFIRMED_MATCHES = {}

# Reset confirmations so editing this cell and rerunning does not keep old decisions.
for match in all_matches:
    match["confirmed"] = False
    match["confirmation_note"] = ""
for project_key, confirmations in CONFIRMED_MATCHES.items():
    if project_key not in matches_by_project:
        raise ValueError("Unknown project key: " + project_key)
    seen = []
    for confirmation in confirmations:
        feature_id = confirmation["osm_feature_id"]
        if feature_id in seen or not confirmation["note"].strip():
            raise ValueError("Each confirmation needs a unique feature and an evidence note")
        seen.append(feature_id)
        matching = [m for m in matches_by_project[project_key] if m["osm_feature_id"] == feature_id]
        if len(matching) != 1:
            raise ValueError("Feature is not a unique candidate: " + feature_id)
        matching[0]["confirmed"] = True
        matching[0]["confirmation_note"] = confirmation["note"]
print("Explicitly confirmed candidate matches:", sum(m["confirmed"] for m in all_matches))

## 6. Build project geometry
`build_project_geometry()` uses only confirmed features:

- Two confirmed substation points: `endpoint_straight_line`, a straight proxy, **not the actual route**.
- One confirmed substation point: `single_confirmed_point`, an incomplete location if another endpoint is missing.
- A confirmed full line: `confirmed_osm_line`, preserving its vertices.
- No usable confirmation or more than two sites: `unresolved`, with null geometry.

For a polygon substation, the point proxy is its first actual boundary vertex, labeled `osm_substation_boundary_vertex_proxy`. It is not a centroid. Full substation polygons and line geometry remain in the candidate/raw files. Complex relation geometry requires further review before making a point proxy.

GridLock's later distance step must use project geometries and closest points, with these approximation labels carried through. This notebook calculates no distances.

In [ ]:
located_projects = []
for project in projects:
    matches = matches_by_project[project["project_key"]]
    geometry, method, point_method = build_project_geometry(matches)
    result = dict(project)
    result["geometry"] = geometry
    result["geometry_method"] = method
    result["point_geometry_method"] = point_method
    result["geometry_is_route_proxy"] = method == "endpoint_straight_line"
    result["candidate_count"] = len(matches)
    result["confirmed_count"] = sum(m["confirmed"] for m in matches)
    result["match_confidence"] = "none"
    for confidence in ["high", "medium", "low"]:
        if any(m["match_confidence"] == confidence for m in matches):
            result["match_confidence"] = confidence
            break
    result["osm_feature_id"] = [m["osm_feature_id"] for m in matches]
    result["osm_feature_name"] = [m["osm_feature_name"] for m in matches]
    result["osm_feature_type"] = [m["osm_feature_type"] for m in matches]
    result["match_method"] = [m["match_method"] for m in matches] or ["manual_review_needed"]
    result["review_status"] = "geometry_confirmed" if geometry else "manual_review_needed"
    located_projects.append(result)

assert len(located_projects) == len(projects)
assert len({p["project_key"] for p in located_projects}) == len(projects)

## 7. Save and summarize
`save_results()` writes the three requested project files plus `<output_prefix>_osm_match_candidates.csv`. That extra table keeps **one row per candidate**, including the OSM geometry and evidence, so uncertain matches are not lost. JSON geometry is stored as text in CSV; the GeoJSON has proper geometry objects.

Confidence counts below are per project, using its strongest candidate. ?Unresolved? counts missing confirmed geometry and can overlap those confidence counts. Initial runs may leave every geometry unresolved; that is preferable to assigning an unverified location. Rerun steps 5?7 after reviewing candidates.

All project CSVs, GeoJSON properties, unresolved rows, and candidate rows preserve `state`. The filename prefix follows the input filename; all generated location outputs go in `data/processed/geo/`.

In [ ]:
df_geo = save_results(located_projects, all_matches, output_prefix=OUTPUT_PREFIX)
print("Total projects:", len(df_geo))
print(df_geo.groupby(["state", "match_confidence"]).size().to_string())
print("Projects with no candidate:", (df_geo["candidate_count"] == 0).sum())
print("Confirmed project geometries:", (df_geo["geometry_method"] != "unresolved").sum())
print("Unresolved:", (df_geo["geometry_method"] == "unresolved").sum())
print("Outputs:")
for suffix in ["_projects_geo.geojson", "_projects_geo.csv", "_unresolved_projects.csv", "_osm_match_candidates.csv"]:
    print(GEO_DIR / (OUTPUT_PREFIX + suffix))

saved = json.loads((GEO_DIR / (OUTPUT_PREFIX + "_projects_geo.geojson")).read_text(encoding="utf-8"))
assert len(saved["features"]) == len(projects)
assert all(f["properties"]["state"] for f in saved["features"])
df_geo[["state", "project_key", "project_name", "candidate_count", "match_confidence", "geometry_method"]].head(15)